# PhotonIQ — Experimental Spectral Learning · v1.0

**Train a mask-aware 1D CNN on real optoelectronic thin-film measurements.** This is a reproducible research notebook, not a website. **No synthetic training data or synthetic fallback.**

**Source**: M. Harth and L. Vesce, *Perovskite-Thin-Films* (Mendeley Data, v3, 2023). [DOI: 10.17632/zt2j5j4v44.3](https://doi.org/10.17632/zt2j5j4v44.3). CC BY 4.0. Corresponding study: *Optoelectronic perovskite film characterization via machine vision* (Solar Energy, 2023).

**Outputs**: source checksum, audited scan-to-thickness joins, group-disjoint train/val/test manifest, train-only scaling, masked CNN training, mean/Ridge baselines, MAE/RMSE/R² and group-bootstrap intervals, validation-film conformal intervals, saved model checkpoint and complete exported results.

**Important**: film thickness is *only a target*. It is never used to divide or otherwise construct optical input features. The model is empirical, **not** claimed physics-informed. `TOP/MID/BOT` scans from the same physical film must stay in the same split. Results are unverified until this notebook runs on the publisher's original experimental ZIP.

In [ ]:
%pip -q install "numpy>=1.24,<3" "pandas>=2,<3" "scikit-learn>=1.3,<2" "openpyxl>=3.1,<4" "joblib>=1.3,<2" "nbformat>=5.9,<6"

## 1 · Acquire the original data

Download the experimental ZIP using **Download All** at [Mendeley Data v3](https://data.mendeley.com/datasets/zt2j5j4v44/3), and upload the unchanged ZIP in the cell below. You may instead supply a path to a copy in Google Drive. The source archive is not committed to Git. The code verifies ZIP format and SHA-256, and refuses malformed or missing experimental files.

If you use a Google Drive ZIP, mount Drive and set `ARCHIVE_IN_DRIVE` to the full filename. This notebook has no fabricated replacement dataset.

In [ ]:
from pathlib import Path
ARCHIVE_IN_DRIVE = ""  # Optional e.g. /content/drive/MyDrive/Datasets/Perovskite-Thin-Films.zip
if ARCHIVE_IN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    archive_path = Path(ARCHIVE_IN_DRIVE)
else:
    from google.colab import files
    print('Upload ORIGINAL Mendeley Download All ZIP (v3):')
    uploads = files.upload()
    names = [name for name in uploads if name.lower().endswith('.zip')]
    if len(names) != 1:
        raise ValueError('Upload exactly ONE source ZIP from Mendeley Data; do not upload synthetic test fixtures.')
    archive_path = Path(names[0]).resolve()
if not archive_path.is_file():
    raise FileNotFoundError(archive_path)
print('Source ZIP selected:', archive_path.name, '| bytes:', archive_path.stat().st_size)

## 2 · Install the versioned PhotonIQ source

These cells are **generated verbatim from the GitHub repository's `src/` modules** by `tools/build_colab.py`. They make this notebook fully usable even before the repository is public. To change methods, edit `src/` and regenerate the notebook; do not manually diverge this file.

In [ ]:
from pathlib import Path
Path('/content/photoniq_runtime/photoniq').mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/photoniq_runtime/photoniq/__init__.py
"""PhotonIQ: reproducible real-data thin-film spectral learning."""
__version__ = '1.0.0'


In [ ]:
%%writefile /content/photoniq_runtime/photoniq/data.py
"""Strict experimental dataset parsing and film-group splits.

The target (film thickness) must NEVER be used to compute model features.
The source dataset belongs to Harth & Vesce and is not redistributed here.
"""
from __future__ import annotations

import hashlib
import re
import shutil
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SOURCE_URL = 'https://data.mendeley.com/datasets/zt2j5j4v44/3'
SOURCE_DOI = '10.17632/zt2j5j4v44.3'
MAX_ARCHIVE_BYTES = 1_500_000_000


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def safe_extract(archive: Path, folder: Path) -> None:
    """Refuse absolute/traversal paths, symlinks, or excessive uncompressed data."""
    folder = folder.resolve()
    folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        total = 0
        for info in z.infolist():
            path = (folder / info.filename).resolve()
            if path != folder and folder not in path.parents:
                raise ValueError(f'Unsafe ZIP member: {info.filename}')
            # POSIX zip external attributes: symbolic-link mode.
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError(f'Symlink refused in ZIP: {info.filename}')
            total += info.file_size
            if total > MAX_ARCHIVE_BYTES:
                raise ValueError('Extracted archive exceeds size safety cap')
        z.extractall(folder)


def unpack_source(archive: Path, output: Path) -> list[Path]:
    """Extract original ZIP and at most two layers of nested source ZIPs."""
    archive = Path(archive)
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        raise ValueError('Provide the actual Mendeley Download All ZIP file.')
    output = Path(output)
    if output.exists():
        shutil.rmtree(output)
    safe_extract(archive, output)
    seen: set[str] = set()
    for _ in range(2):
        nested = [p for p in output.rglob('*') if p.suffix.lower() == '.zip' and str(p) not in seen]
        if not nested:
            break
        for p in nested:
            seen.add(str(p))
            if zipfile.is_zipfile(p):
                safe_extract(p, p.parent / (p.stem + '_unpacked'))
    return sorted(p for p in output.rglob('*') if p.is_file())


def sample_key(text: str) -> str:
    text = str(text).strip().lower().replace('-', '_').replace(' ', '_')
    found = re.search(r'(?<!\d)(\d+_\d+_\d+_\d+(?:_\d+)?)(?!\d)', text)
    if found:
        return found.group(1)
    return re.sub(r'[^a-z0-9]+', '_', re.sub(r'top|middle|mid|bottom|bot', '', text)).strip('_')


def position_key(filename: str) -> str | None:
    u = str(filename).upper()
    found = [p for p, terms in {
        'TOP': ('TOP',), 'MID': ('MID', 'MIDDLE'), 'BOT': ('BOT', 'BOTTOM')
    }.items() if any(term in u for term in terms)]
    return found[0] if len(found) == 1 else None


def parse_thickness_workbook(path: Path) -> pd.DataFrame:
    """Exact published 4-row, transposed Excel layout. No guessed row labels."""
    raw = pd.read_excel(path, header=None, skiprows=1, nrows=4, engine='openpyxl')
    if raw.shape[0] != 4:
        raise ValueError(f'Unsupported thickness workbook layout: {path.name}; shape={raw.shape}')
    frame = raw.T
    frame.columns = ['sample', 'TOP', 'MID', 'BOT']
    frame = frame.iloc[1:]  # published source skips first transposed header row
    records = []
    for _, row in frame.iterrows():
        sample_id = sample_key(row['sample'])
        if not sample_id:
            continue
        for position in ('TOP', 'MID', 'BOT'):
            value = pd.to_numeric(row[position], errors='coerce')
            if np.isfinite(value) and 5 <= value <= 5000:
                records.append(dict(sample_id=sample_id, position=position,
                                    thickness_nm=float(value), label_source=path.name))
    if not records:
        raise ValueError(f'No usable thickness labels in {path}')
    return pd.DataFrame.from_records(records)


def parse_spectrum(path: Path, min_points: int = 30) -> np.ndarray | None:
    """Read experimental UV–Vis TXT; first 2 lines are instrument headers."""
    points = []
    for line in Path(path).read_text(encoding='utf-8', errors='replace').splitlines()[2:]:
        parts = re.split(r'[\s,;]+', line.strip())
        if len(parts) < 2:
            continue
        try:
            w, intensity = float(parts[0]), float(parts[1])
        except ValueError:
            continue
        if np.isfinite(w) and np.isfinite(intensity):
            points.append((w, intensity))
    if len(points) < min_points:
        return None
    data = pd.DataFrame(points, columns=['wavelength', 'signal']).groupby('wavelength', as_index=False).mean()
    arr = data.to_numpy(dtype=np.float64)
    if len(arr) < min_points or arr[0, 0] < 180 or arr[-1, 0] > 2500 or np.ptp(arr[:, 0]) < 80:
        return None
    return arr


@dataclass
class ExperimentalData:
    X: np.ndarray
    y: np.ndarray
    group: np.ndarray
    position: np.ndarray
    path: list[str]
    wavelengths: np.ndarray
    audit: dict


def load_experimental(source_dir: Path, grid_points: int = 128, min_match_fraction: float = .65) -> ExperimentalData:
    root = Path(source_dir)
    workbooks = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.xlsx'
                 and 'thickness' in p.name.lower()]
    if not workbooks:
        raise FileNotFoundError('Thickness XLSX workbooks missing; use published original data archive.')
    labels = pd.concat([parse_thickness_workbook(p) for p in sorted(workbooks)], ignore_index=True)
    # Conflicting labels for the same film/position are disallowed; never silently discard.
    grouped = labels.groupby(['sample_id', 'position']).thickness_nm
    if (grouped.max() - grouped.min() > 0.01).any():
        raise ValueError('Conflicting thickness labels detected for the same sample/position.')
    labels = labels.drop_duplicates(subset=['sample_id', 'position'])

    # Only files explicitly located in a UV-Vis directory are considered spectra.
    # Do NOT search PL data or other instrument signals as fallbacks.
    raw_files = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.txt']
    uv_files = [p for p in raw_files if position_key(p.stem) is not None and
                any('uvvis' in str(part).lower().replace('_', '').replace('-', '') for part in p.parts)]
    if not uv_files:
        raise FileNotFoundError('No TOP/MID/BOT TXT files under UVVis directory; inspect source archive layout.')
    spectra = []
    for path in sorted(uv_files):
        arr = parse_spectrum(path)
        if arr is not None:
            spectra.append(dict(sample_id=sample_key(path.stem), position=position_key(path.stem),
                                path=str(path.relative_to(root)), array=arr))
    if not spectra:
        raise ValueError('All selected UV–Vis files failed numeric validation')
    spec = pd.DataFrame(spectra)
    matched = spec.merge(labels, on=['sample_id', 'position'], how='inner', validate='many_to_one')
    coverage = len(matched) / len(spec)
    if coverage < min_match_fraction:
        raise ValueError(f'Too few matched spectra: {len(matched)}/{len(spec)} ({coverage:.1%}); inspect IDs and workbooks.')
    if matched.sample_id.nunique() < 15:
        raise ValueError('Fewer than 15 distinct labelled films; grouped evaluation would be unreliable.')
    matched = matched.sort_values(['sample_id', 'position', 'path']).reset_index(drop=True)
    lo = max(float(arr[0, 0]) for arr in matched['array'])
    hi = min(float(arr[-1, 0]) for arr in matched['array'])
    if hi - lo < 80:
        raise ValueError('Insufficient shared wavelength span for interpolation.')
    grid = np.linspace(lo, hi, grid_points, dtype=np.float32)
    X = np.stack([np.interp(grid, arr[:, 0], arr[:, 1]) for arr in matched['array']]).astype('float32')
    y = matched.thickness_nm.to_numpy(dtype='float32')
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError('Nonfinite feature/target values after preprocessing')
    audit = dict(dataset_doi=SOURCE_DOI, spectra_found=len(spec), spectra_matched=len(matched),
                 match_fraction=round(coverage, 5), films=int(matched.sample_id.nunique()),
                 workbooks=[p.name for p in workbooks],
                 missing_spectra=int(len(spec) - len(matched)),
                 wavelength_min_nm=round(lo, 4), wavelength_max_nm=round(hi, 4),
                 instrument_signal_units='original UV-Vis signal units; not thickness-normalised',
                 target_units='nm', original_data_type='EXPERIMENTAL')
    return ExperimentalData(X, y, matched.sample_id.to_numpy(dtype=str),
                            matched.position.to_numpy(dtype=str), matched.path.tolist(), grid, audit)


def grouped_split(groups: np.ndarray, seed: int = 42) -> dict[str, np.ndarray]:
    groups = np.asarray(groups)
    indices = np.arange(len(groups))
    splitter = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed)
    tv, test = next(splitter.split(indices, groups=groups))
    second = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed + 1)
    t, v = next(second.split(tv, groups=groups[tv]))
    splits = {'train': tv[t], 'val': tv[v], 'test': test}
    sets = [set(groups[idx]) for idx in splits.values()]
    assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]), 'Group leakage'
    assert all(len(g) >= 3 for g in sets), 'Insufficient independent films in one split'
    return splits


In [ ]:
%%writefile /content/photoniq_runtime/photoniq/model.py
"""Compact mask-aware convolutional regression; no simulation-based pretraining."""
from __future__ import annotations

import numpy as np
import torch
from torch import nn


class MaskAwareSpectralCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(2, 16, 7, padding=3), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 5, padding=2), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 32, 5, padding=2), nn.GELU(), nn.AdaptiveAvgPool1d(8),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 8, 64), nn.GELU(),
                                  nn.Dropout(.15), nn.Linear(64, 1))

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(features)).squeeze(1)


def mask_spectra(x: torch.Tensor, keep: int | None = None,
                 rng: np.random.Generator | None = None,
                 indices: np.ndarray | None = None) -> torch.Tensor:
    """Two channels: standardized measured intensity (zeros if hidden) and measurement mask."""
    rng = rng or np.random.default_rng()
    batch, points = x.shape
    if indices is not None:
        selected = np.asarray(indices, dtype=int)
        if not len(selected) or np.any(selected < 0) or np.any(selected >= points):
            raise ValueError('Invalid wavelength indices')
        masks = np.zeros((batch, points), dtype='float32')
        masks[:, selected] = 1
    elif keep is None:
        counts = rng.choice([16, 32, 64, 96, points], size=batch, p=[.10, .15, .2, .2, .35])
        masks = np.zeros((batch, points), dtype='float32')
        for i, count in enumerate(counts):
            masks[i, rng.choice(points, size=min(int(count), points), replace=False)] = 1
    else:
        if keep < 1 or keep > points:
            raise ValueError('keep must be in 1..n_wavelengths')
        chosen = np.unique(np.rint(np.linspace(0, points - 1, keep)).astype(int))
        masks = np.zeros((batch, points), dtype='float32')
        masks[:, chosen] = 1
    m = torch.tensor(masks, device=x.device, dtype=x.dtype)
    return torch.stack([x * m, m], dim=1)


In [ ]:
%%writefile /content/photoniq_runtime/photoniq/metrics.py
"""Experimental metrics and film-level bootstrap uncertainty."""
from __future__ import annotations
import math
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def score(y: np.ndarray, pred: np.ndarray, groups: np.ndarray) -> dict:
    groups = np.asarray(groups)
    errors = np.abs(y - pred)
    film_errors = [float(np.mean(errors[groups == g])) for g in np.unique(groups)]
    return dict(MAE_nm=float(mean_absolute_error(y, pred)),
                RMSE_nm=float(np.sqrt(mean_squared_error(y, pred))),
                film_macro_MAE_nm=float(np.mean(film_errors)),
                R2=float(r2_score(y, pred)) if len(y) > 1 else float('nan'),
                scans=int(len(y)), independent_films=int(len(film_errors)))


def film_bootstrap_mae_ci(y: np.ndarray, pred: np.ndarray, groups: np.ndarray,
                          n_boot: int = 1000, seed: int = 42) -> tuple[float, float]:
    """Resample independent films (not scan rows); report scan-weighted MAE distribution."""
    unique = np.unique(groups)
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        draw = rng.choice(unique, size=len(unique), replace=True)
        chosen = np.concatenate([np.where(groups == g)[0] for g in draw])
        values.append(float(np.abs(y[chosen] - pred[chosen]).mean()))
    return tuple(float(x) for x in np.quantile(values, [.025, .975]))


def film_conformal_width(y_val: np.ndarray, pred_val: np.ndarray, groups: np.ndarray,
                         alpha: float = .1) -> tuple[float, dict]:
    """Group maximum-error conformal width; large if calibration set is small.

    Groupwise calibration gives an interval covering all *positions* on a new film
    under film-level exchangeability; it is not a guaranteed calibrated interval
    under arbitrary domain shift. No test labels influence the width.
    """
    unique = np.unique(groups)
    residuals = np.abs(y_val - pred_val)
    scores = np.sort(np.array([residuals[groups == g].max() for g in unique]))
    n = len(scores)
    rank = math.ceil((n + 1) * (1 - alpha))
    if rank > n:
        return float('inf'), dict(calibration_films=n, nominal_coverage=1-alpha,
                                  finite_sample_width_available=False,
                                  note='Too few calibration films for a finite distribution-free width')
    return float(scores[rank - 1]), dict(calibration_films=n,
                                         nominal_coverage=1-alpha,
                                         finite_sample_width_available=True)


In [ ]:
%%writefile /content/photoniq_runtime/photoniq/train.py
"""Reproducible training/evaluation on real thin-film measurements.

No test-informed tuning. No synthetic fallback. No target-derived feature computation.
"""
from __future__ import annotations

import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from . import __version__
from .data import file_sha256, grouped_split, load_experimental, unpack_source
from .metrics import film_bootstrap_mae_ci, film_conformal_width, score
from .model import MaskAwareSpectralCNN, mask_spectra


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.set_num_threads(min(4, os.cpu_count() or 2))


def save_json(path: Path, data: dict | list):
    path.write_text(json.dumps(data, indent=2, allow_nan=False, default=str), encoding='utf-8')


def fit_cnn(X: np.ndarray, y: np.ndarray, splits: dict,
            epochs: int = 150, patience: int = 20, seed: int = 42,
            device: str = 'cpu', batch_size: int = 16) -> tuple:
    # X: training-standardized spectra; y: training-standardized targets
    net = MaskAwareSpectralCNN().to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.SmoothL1Loss(beta=0.6)
    tr, va = splits['train'], splits['val']
    rng = np.random.default_rng(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(X[tr]), torch.from_numpy(y[tr])),
                        batch_size=batch_size, shuffle=True, drop_last=False,
                        generator=torch.Generator().manual_seed(seed))
    validation_x = torch.from_numpy(X[va]).to(device)
    validation_y = torch.from_numpy(y[va]).to(device)
    best = float('inf'); state = None; idle = 0; history = []
    for epoch in range(1, epochs + 1):
        net.train(); losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = net(mask_spectra(xb, rng=rng))
            loss = loss_fn(pred, yb)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 2.0)
            optimizer.step()
            losses.append(float(loss.item()))
        net.eval()
        with torch.no_grad():
            vpred = net(mask_spectra(validation_x, keep=X.shape[1]))
            vloss = float(loss_fn(vpred, validation_y).item())
        history.append(dict(epoch=epoch, train_loss=np.mean(losses), val_loss=vloss))
        if vloss + 1e-5 < best:
            best, idle = vloss, 0
            state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            idle += 1
        if idle >= patience:
            break
    assert state is not None
    net.load_state_dict(state)
    return net, pd.DataFrame(history), best


def cnn_predict(net, X: np.ndarray, indices: np.ndarray, device: str,
                ymean: float, ystd: float, keep: int | None = None) -> np.ndarray:
    net.eval()
    x = torch.from_numpy(X[indices]).to(device)
    with torch.no_grad():
        result = net(mask_spectra(x, keep=keep or X.shape[1])).detach().cpu().numpy()
    return result * ystd + ymean


def run_experiment(archive: str | Path, output: str | Path, *,
                   epochs: int = 150, patience: int = 20,
                   seed: int = 42, n_boot: int = 1000,
                   device: str = 'auto', grid_points: int = 128,
                   min_match_fraction: float = .65) -> dict:
    archive = Path(archive).expanduser().resolve()
    output = Path(output).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    real_device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
    if real_device == 'cuda' and not torch.cuda.is_available():
        raise ValueError('CUDA requested but no GPU is available')
    t0 = time.perf_counter()
    print('Dataset: EXPERIMENTAL Mendeley Harth–Vesce v3 (no synthetic fallback)')
    print('Source archive SHA256:', file_sha256(archive))
    files = unpack_source(archive, output / 'extracted_original_dataset')
    dataset = load_experimental(output / 'extracted_original_dataset', grid_points, min_match_fraction)
    print('Files:', len(files), '| Matched scans:', len(dataset.y),
          '| Independent films:', len(np.unique(dataset.group)))
    splits = grouped_split(dataset.group, seed)
    for name, ix in splits.items():
        print(name, ':', len(ix), 'scans /', len(np.unique(dataset.group[ix])), 'independent films')

    # Film-level split is fixed before fit; fit scaler/target scaling on TRAIN ONLY.
    scaler = StandardScaler().fit(dataset.X[splits['train']])
    X = scaler.transform(dataset.X).astype('float32')
    ymean = float(dataset.y[splits['train']].mean())
    ystd = float(dataset.y[splits['train']].std())
    if ystd < 1e-6:
        raise ValueError('Insufficient training target variation')
    yscaled = ((dataset.y - ymean) / ystd).astype('float32')

    split_by_index = np.empty(len(dataset.y), dtype=object)
    for name, ix in splits.items():
        split_by_index[ix] = name
    manifest = pd.DataFrame(dict(sample_id=dataset.group, position=dataset.position,
                                 thickness_nm=dataset.y, source_path=dataset.path,
                                 split=split_by_index))
    manifest.to_csv(output / 'split_manifest.csv', index=False)
    save_json(output / 'data_audit.json', dataset.audit)

    # Baselines: hyperparameters fixed in protocol; never chosen using test performance.
    mean_baseline = DummyRegressor(strategy='mean').fit(X[splits['train']], dataset.y[splits['train']])
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
    ridge.fit(dataset.X[splits['train']], dataset.y[splits['train']])

    net, history, best_val = fit_cnn(X, yscaled, splits, epochs, patience, seed, real_device)
    history.to_csv(output / 'learning_curve.csv', index=False)

    # Calibrate on validation films, using maximum position error per physical film.
    val_ix = splits['val']; test_ix = splits['test']
    val_pred = cnn_predict(net, X, val_ix, real_device, ymean, ystd)
    width, conformal_info = film_conformal_width(dataset.y[val_ix], val_pred,
                                                  dataset.group[val_ix], alpha=.1)
    truth = dataset.y[test_ix]; test_groups = dataset.group[test_ix]
    models = {'Training-mean baseline': mean_baseline.predict(X[test_ix]),
              'Ridge baseline': ridge.predict(dataset.X[test_ix])}
    for keep in [16, 32, 64, grid_points]:
        if keep <= grid_points:
            models[f'Mask-aware CNN ({keep}/{grid_points} wavelengths)'] = (
                cnn_predict(net, X, test_ix, real_device, ymean, ystd, keep=keep))

    rows = []; predictions = manifest.iloc[test_ix].copy()
    for name, pred in models.items():
        s = score(truth, pred, test_groups)
        low, high = film_bootstrap_mae_ci(truth, pred, test_groups, n_boot, seed)
        rows.append(dict(model=name, **s, MAE_95pct_cluster_bootstrap_low_nm=low,
                         MAE_95pct_cluster_bootstrap_high_nm=high))
        if 'wavelengths)' in name and name.endswith(f'({grid_points}/{grid_points} wavelengths)'):
            predictions['cnn_full_prediction_nm'] = pred
        if name == 'Ridge baseline':
            predictions['ridge_prediction_nm'] = pred
    results = pd.DataFrame(rows)
    results.to_csv(output / 'test_metrics.csv', index=False)
    predictions.to_csv(output / 'test_predictions.csv', index=False)

    full = models[f'Mask-aware CNN ({grid_points}/{grid_points} wavelengths)']
    if np.isfinite(width):
        predictions['cnn_interval_low_nm'] = full - width
        predictions['cnn_interval_high_nm'] = full + width
        position_coverage = float(np.mean(np.abs(truth-full) <= width))
        film_coverage = float(np.mean([
            bool(np.all(np.abs(truth[test_groups == group] - full[test_groups == group]) <= width))
            for group in np.unique(test_groups)]))
    else:
        position_coverage = None
        film_coverage = None
    conformal_info.update(dict(half_width_nm=width if np.isfinite(width) else None,
                               test_position_coverage=position_coverage,
                               test_film_coverage=film_coverage,
                               note='Groupwise exchangeability assumption; not valid under unrestricted domain shift'))
    predictions.to_csv(output / 'test_predictions.csv', index=False)
    save_json(output / 'conformal_evaluation.json', conformal_info)

    torch.save(dict(model_state=net.cpu().state_dict(), architecture='MaskAwareSpectralCNN',
                    feature_grid_nm=dataset.wavelengths.tolist(),
                    feature_mean=scaler.mean_.tolist(), feature_scale=scaler.scale_.tolist(),
                    target_mean_nm=ymean, target_sd_nm=ystd, training_seed=seed,
                    dataset_source_sha256=file_sha256(archive)), output / 'photoniq_cnn.pt')
    import joblib
    joblib.dump(ridge, output / 'ridge_baseline.joblib')
    metadata = dict(software_version=__version__, dataset_doi=dataset.audit['dataset_doi'],
                    dataset_type='experimentally measured UV-Vis + measured thickness',
                    dataset_sha256=file_sha256(archive), source_licence='CC BY 4.0',
                    seed=seed, device=real_device, python=platform.python_version(),
                    torch=torch.__version__, numpy=np.__version__, sklearn=sklearn.__version__,
                    independent_films=int(len(set(dataset.group))), total_scans=len(dataset.y),
                    split_film_counts={k:len(set(dataset.group[v])) for k,v in splits.items()},
                    training_epochs=int(len(history)), best_validation_loss=float(best_val),
                    seconds=float(time.perf_counter()-t0),
                    limitations=['Single experimental dataset; domain shift not tested',
                                 'Small independent-film counts; uncertainty intervals may be wide',
                                 'Model is empirical, not physics-informed',
                                 'Sparse masks represent observed channels, not active wavelength selection'])
    save_json(output / 'run_metadata.json', metadata)
    print(results.to_string(index=False, float_format=lambda a: f'{a:.4f}'))
    print('Outputs:', output)
    return dict(data=dataset.audit, metrics=results.to_dict(orient='records'),
                conformal=conformal_info, metadata=metadata)


## 3 · Audit, train, evaluate

Default: independent-film 60/20/20 train/validation/test split, 128 wavelength bins, up to 150 epochs with validation early stopping, CPU/GPU auto selection. The test split is never used to choose architecture or epoch. Small datasets may produce wide confidence intervals. To adjust training for a quick dry run, set `EPOCHS = 3`; do **not** publish that as a scientific result.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/photoniq_runtime')
from photoniq.train import run_experiment

EPOCHS = 150
PATIENCE = 20
SEED = 42
BOOTSTRAP_ITERATIONS = 1000
run_dir = Path('/content/PhotonIQ_Experimental_Results')
report = run_experiment(archive_path, run_dir, epochs=EPOCHS, patience=PATIENCE,
                        seed=SEED, n_boot=BOOTSTRAP_ITERATIONS, device='auto')
print('Independent films:', report['metadata']['independent_films'])
print('Dataset SHA256:', report['metadata']['dataset_sha256'])

## 4 · Inspect independent test performance

Interpret the CNN alongside conventional baselines; don't declare it superior unless results support that. The sparse-channel runs use uniformly spaced *fixed* wavelength masks, NOT an adaptive acquisition policy. The conformal interval assumes exchangeability at the film level and is not a certificate of accuracy on other materials or instruments.

In [ ]:
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(run_dir/'test_metrics.csv')
manifest = pd.read_csv(run_dir/'split_manifest.csv')
print('Group-level distribution (FILMS, not scans):')
display(manifest.groupby('split').sample_id.nunique())
display(metrics)
print('Group-conformal evaluation:')
print((run_dir/'conformal_evaluation.json').read_text())

In [ ]:
import matplotlib.pyplot as plt
preds = pd.read_csv(run_dir/'test_predictions.csv')
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(preds.thickness_nm, preds.cnn_full_prediction_nm, alpha=.7)
lo = min(preds.thickness_nm.min(), preds.cnn_full_prediction_nm.min())
hi = max(preds.thickness_nm.max(), preds.cnn_full_prediction_nm.max())
ax.plot([lo,hi], [lo,hi], '--', color='black', lw=1)
ax.set(xlabel='Measured film thickness (nm)', ylabel='CNN prediction (nm)',
       title='Held-out experimental films — full UV–Vis spectrum')
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(run_dir/'experimental_test_scatter.png', dpi=180)
plt.show()

## 5 · Export all results

The ZIP contains predictions, split manifest, metrics, dataset audit, conformal analysis, model weights, and environment metadata. **It does not include the source's raw experimental data**. Before publishing any GitHub performance table, inspect the run audit and independent film counts; retrain if the parser reports inadequate matches.

In [ ]:
from google.colab import files
import shutil
bundle_path = shutil.make_archive('/content/PhotonIQ_Experimental_Results', 'zip', root_dir=run_dir)
print('Export:', bundle_path)
files.download(bundle_path)

### Responsible interpretation

- Not enough evidence to claim a physics-informed method, active wavelength selection, or cross-material generalization.
- Group-disjoint test scans are still only from the same published experimental collection; external validation remains future work.
- Report per-film (not only per-row) accuracy, confidence intervals, source DOI, and data SHA256.
- If the CNN performs worse than Ridge, report that result.

For the methodology, see `docs/DATASET.md`, `docs/MODEL_CARD.md`, and `docs/EXPERIMENT_PROTOCOL.md` in the GitHub-ready project package.